# Branch the brain — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will fork your
factory database, change a machine on the fork, prove your original is untouched, and throw
the fork away.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL
editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA`
are set and the factory is loaded on your branch.


## Setup — connect to your branch


In [ ]:
import os, sys, time
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import client, branch_name, branch_endpoint, wait_active
from databricks.sdk.service.postgres import Branch, BranchSpec
import psycopg

PROJECT = os.environ['LB_PROJECT']
MINE    = os.environ['LB_BRANCH']
EXP     = MINE + '-exp'
SCHEMA  = os.environ.get('LB_SCHEMA', 'factory')
MACHINE = 'M-204'
w = client()

def connect_branch(branch):
    host, eid, state = branch_endpoint(w, PROJECT, branch)
    if not host: raise RuntimeError(f'branch {branch} not ready ({state})')
    cred = w.postgres.generate_database_credential(endpoint=f'{branch_name(PROJECT, branch)}/endpoints/{eid}')
    conn = psycopg.connect(host=host, port=5432, dbname='databricks_postgres',
                           user=w.current_user.me().user_name, password=cred.token,
                           sslmode='require', autocommit=True)
    with conn.cursor() as cur: cur.execute(f'SET search_path TO {SCHEMA}, public')
    return conn
print('connected as', w.current_user.me().user_name)


## Step 1 — Look at M-204 on your branch

`M-204` is a stamping press. Read its current state; it should be `healthy`. The SQL:

```sql
SELECT machine_id, status, health_pct FROM factory.machine WHERE machine_id = 'M-204';
```


In [ ]:
with connect_branch(MINE) as c, c.cursor() as cur:
    cur.execute("SELECT machine_id, status, health_pct FROM machine WHERE machine_id=%s", [MACHINE])
    print('your branch, before:', cur.fetchone())


## Step 2 — Fork your branch (zero-copy)

Forking is a control-plane operation, not SQL, so this step uses the SDK. It is copy-on-write:
no data copied, API back in under a second, endpoint connectable in about two seconds. The
equivalent CLI, if you prefer the terminal:

```bash
databricks postgres create-branch --profile "$LB_PROFILE" \
  --parent projects/$LB_PROJECT \
  --json '{"branch_id": "'$LB_BRANCH'-exp", "spec": {"source_branch": "projects/'$LB_PROJECT'/branches/'$LB_BRANCH'"}}'
```


In [ ]:
try:
    w.postgres.update_branch(name=branch_name(PROJECT, EXP),
        branch=Branch(spec=BranchSpec(is_protected=False)), update_mask='spec.is_protected')
    w.postgres.delete_branch(name=branch_name(PROJECT, EXP)); time.sleep(2)
except Exception: pass
t0 = time.time()
w.postgres.create_branch(parent=f'projects/{PROJECT}',
    branch=Branch(spec=BranchSpec(source_branch=branch_name(PROJECT, MINE), no_expiry=True)),
    branch_id=EXP, replace_existing=False)
api_ms=(time.time()-t0)*1000; waited=wait_active(w, PROJECT, EXP)
print(f'fork API {api_ms:.0f} ms; endpoint ACTIVE in ~{waited:.1f}s')


## Step 3 — Diverge on the fork, prove production is untouched

Change `M-204` on the fork only, then read it from both branches. The SQL that runs on the fork:

```sql
UPDATE factory.machine SET status = 'derated', health_pct = 70.0 WHERE machine_id = 'M-204';
```


In [ ]:
with connect_branch(EXP) as c, c.cursor() as cur:
    cur.execute("UPDATE machine SET status='derated', health_pct=70.0 WHERE machine_id=%s", [MACHINE])
    cur.execute("SELECT machine_id, status, health_pct FROM machine WHERE machine_id=%s", [MACHINE])
    print('fork, after change:', cur.fetchone())
with connect_branch(MINE) as c, c.cursor() as cur:
    cur.execute("SELECT machine_id, status, health_pct FROM machine WHERE machine_id=%s", [MACHINE])
    print('your branch, still: ', cur.fetchone())


## Step 4 — Throw the fork away

Discard the experiment branch. Only changed pages were ever written, so this leaves zero
storage behind and your branch is intact.


In [ ]:
w.postgres.update_branch(name=branch_name(PROJECT, EXP),
    branch=Branch(spec=BranchSpec(is_protected=False)), update_mask='spec.is_protected')
w.postgres.delete_branch(name=branch_name(PROJECT, EXP))
print('discarded', EXP)
print('branches now:', [b.branch_id for b in w.postgres.list_branches(parent=f'projects/{PROJECT}')])


## What you learned

Branching a live database is seconds-scale and near-free, because it is copy-on-write. You
forked, diverged, and discarded without touching your source data. Next: **Break it, then
rewind** points this same machinery at the past.
